# Example: Full Document Retrieval Tool

To showcase tool use, we are going to consider a full document retrieval tool:

In [ ]:
%load_ext ipython_async

In [ ]:
%%asynccmd
mkdir .\vectorstore\postgres &
initdb -D .\vectorstore\postgres --encoding=UTF8 --locale=en_US.UTF-8 &
pg_ctl -D .\vectorstore\postgres -l logfile start

In [ ]:
import os

In [ ]:
db_name = "vector_db"
user = "acisse"

In [ ]:
from llama_index.vector_stores.postgres import PGVectorStore

vector_store = PGVectorStore.from_params(
    database=db_name,
    host="localhost",
    password=os.getenv("PSQL_PASSWORD"),
    port=5432,
    user=user,
    table_name="physics_docs",
    embed_dim=384,
    hybrid_search=True
)

In [ ]:
from llama_index.embeddings.langchain import LangchainEmbedding
from langchain.embeddings import HuggingFaceEmbeddings

# Load MiniLM embedding model (fast & efficient)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Wrap the model for LlamaIndex
embed_model = LangchainEmbedding(embedding_model)

Finally, we can restore the index:

In [ ]:
from pathlib import Path
from llama_index.core import StorageContext, load_index_from_storage

# Load the stored index
index_save_path = Path("assets", "llama_vector_search", "physics_documents_index")
storage_context = StorageContext.from_defaults(persist_dir=index_save_path, vector_store=vector_store)
index = load_index_from_storage(storage_context, embed_model=embed_model)

And load our secrets into the environment:

In [ ]:
from dotenv import load_dotenv
from utils import display_citation
load_dotenv()

## Modifying the Query Engine

First, we need to modify the query engine, so that the metadata from the chunked files is returned with the query output.

In [ ]:
from llama_index.core.base.base_query_engine import BaseQueryEngine
from llama_index.core.base.response.schema import RESPONSE_TYPE
from llama_index.core.schema import NodeWithScore, QueryBundle
from typing import Any, List, Optional, Sequence, Type
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.base.base_retriever import BaseRetriever
from llama_index.core.settings import Settings
from llama_index.core.callbacks.schema import CBEventType, EventPayload
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.response_synthesizers.base import BaseSynthesizer

In [ ]:
class MetadataRetrieverQueryEngine(BaseQueryEngine):
    """
    A custom retriever query engine that appends metadata from retrieved nodes
    into the final synthesized response.
    """
    def __init__(
        self,
        retriever: BaseRetriever,
        response_synthesizer: Optional[BaseSynthesizer] = None,
        node_postprocessors: Optional[List[BaseNodePostprocessor]] = None,
        callback_manager: Optional[Any] = None,
    ) -> None:
        self._retriever = retriever
        self._response_synthesizer = response_synthesizer or get_response_synthesizer(
            llm=Settings.llm,
            callback_manager=callback_manager or Settings.callback_manager,
        )
        self._node_postprocessors = node_postprocessors or []
        callback_manager = callback_manager or self._response_synthesizer.callback_manager
        for node_postprocessor in self._node_postprocessors:
            node_postprocessor.callback_manager = callback_manager
        super().__init__(callback_manager=callback_manager)

    def _apply_node_postprocessors(
        self, nodes: List[NodeWithScore], query_bundle: QueryBundle
    ) -> List[NodeWithScore]:
        for node_postprocessor in self._node_postprocessors:
            nodes = node_postprocessor.postprocess_nodes(nodes, query_bundle=query_bundle)
        return nodes
    
    def _get_prompt_modules(self) -> dict:
        # This method returns the prompt modules used by the query engine.
        # Here, we simply include the response synthesizer.
        return {"response_synthesizer": self._response_synthesizer}

    def retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        nodes = self._retriever.retrieve(query_bundle)
        return self._apply_node_postprocessors(nodes, query_bundle=query_bundle)

    async def aretrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        nodes = await self._retriever.aretrieve(query_bundle)
        return self._apply_node_postprocessors(nodes, query_bundle=query_bundle)

    @staticmethod 
    def _parse_metadata(node) -> str:
        if not  hasattr(node, "metadata"):
            return ""
        metadata = node.metadata
        return (f"File_name: {Path(metadata['file_path']).parent.stem}\n"
                f"Last modified: {Path(metadata['last_modified_date'])}\n"
                f"File size: {metadata['file_size']}\n"
                f"Node Start char index: {node.start_char_idx}\n"
                f"Node End char index: {node.end_char_idx}\n")

    def synthesize(
        self,
        query_bundle: QueryBundle,
        nodes: List[NodeWithScore],
        additional_source_nodes: Optional[Sequence[NodeWithScore]] = None,
    ) -> RESPONSE_TYPE:
        # Generate the base answer using the underlying synthesizer.
        base_response = self._response_synthesizer.synthesize(
            query=query_bundle,
            nodes=nodes,
            additional_source_nodes=additional_source_nodes,
        )
        # Convert base response to a string (or extract the text if available)
        base_response_text = base_response.response if hasattr(base_response, "response") else str(base_response)

        # Collect metadata from each retrieved node.
        metadata_str = ""
        for node_with_score in nodes:
            node = node_with_score.node
            metadata_str += f"Document Metadata:\n" + self._parse_metadata(node)

        # Create the final response text
        final_response_text = (
            base_response_text
            + "\n\nRetrieved Chunk Metadata:\n"
            + metadata_str
        )
        
        # Wrap the final response text in the same type as base_response.
        final_response_obj = type(base_response)(response=final_response_text)
        return final_response_obj

    async def asynthesize(
        self,
        query_bundle: QueryBundle,
        nodes: List[NodeWithScore],
        additional_source_nodes: Optional[Sequence[NodeWithScore]] = None,
    ) -> RESPONSE_TYPE:
        base_response = await self._response_synthesizer.asynthesize(
            query=query_bundle,
            nodes=nodes,
            additional_source_nodes=additional_source_nodes,
        )
        base_response_text = base_response.response if hasattr(base_response, "response") else str(base_response)
        
        metadata_str = ""
        for node_with_score in nodes:
            node = node_with_score.node
            metadata_str += f"Document Metadata:\n" + self._parse_metadata(node)

        final_response_text = (
            base_response_text
            + "\n\nRetrieved Chunk Metadata:\n"
            + metadata_str
        )
        final_response_obj = type(base_response)(response=final_response_text)
        return final_response_obj

    def _query(self, query_bundle: QueryBundle) -> RESPONSE_TYPE:
        # Retrieve nodes for the query.
        nodes = self.retrieve(query_bundle)
        # Synthesize a response that includes metadata.
        return self.synthesize(query_bundle, nodes)

    async def _aquery(self, query_bundle: QueryBundle) -> RESPONSE_TYPE:
        nodes = await self.aretrieve(query_bundle)
        return await self.asynthesize(query_bundle, nodes)

    @property
    def retriever(self) -> BaseRetriever:
        return self._retriever


In [ ]:
from llama_index.vector_stores.postgres import PGVectorStore
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.postprocessor import SimilarityPostprocessor

# Assume you have an index already loaded.
retriever = index.as_retriever(similarity_top_k=5)
response_synthesizer = get_response_synthesizer()
node_postprocessors = [SimilarityPostprocessor(similarity_cutoff=0.4)]

In [ ]:
query_engine_with_metadata = MetadataRetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=node_postprocessors,
)

response = query_engine_with_metadata.query("Who was Juan Maldacena?")
display_citation(response.response)


In [ ]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o")
Settings.llm = llm
basic_query_engine = index.as_query_engine()
response = basic_query_engine.query("Who was Juan Maldacena?")
display_citation(response.response)

## Basic Agent

First let's have a look at how the basic engine responds to queries including questions about the metadata. The expectation is, that the agent will be unable to access the metadata and return inadequate responses:

In [ ]:
from llama_index.core.agent import StructuredPlannerAgent, FunctionCallingAgentWorker
from llama_index.core.tools import QueryEngineTool
from llama_index.core.tools import ToolMetadata
import nest_asyncio

In [ ]:
nest_asyncio.apply()

# Create a query engine from our index
query_engine = index.as_query_engine()

# Wrap the query engine in a QueryEngineTool
query_engine_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name="vector_index",
        description="A tool for retrieving information from a physics document index."
    )
)

# Create a function-calling worker for executing retrieval tasks
worker = FunctionCallingAgentWorker.from_tools(
    [query_engine_tool], 
    llm=llm, 
    verbose=True,
    allow_parallel_tool_calls=False
)

# Initialize the structured planner agent with the worker
structured_planner = StructuredPlannerAgent(
    worker, 
    tools=[query_engine_tool], 
    verbose=True
)

# Test with a complex question
response = structured_planner.chat("What are the names of the files contianing information about Juan Maldacena according to the metadata?")
display_citation(response.response)


## Metadata Aware Agent

Next, we will use our metadata infused query agent and look at how the agent will retrieve the file name:

In [ ]:
nest_asyncio.apply()

# Wrap the query engine in a QueryEngineTool
query_engine_tool = QueryEngineTool(
    query_engine=query_engine_with_metadata,
    metadata=ToolMetadata(
        name="vector_index",
        description="A tool for retrieving information from a physics document index. Synthesized response includes"
        " metadata about the files, allowing to learn their location and creation data, as well as the chunks location in the file"
    )
)

# Create a function-calling worker for executing retrieval tasks
worker = FunctionCallingAgentWorker.from_tools(
    [query_engine_tool], 
    llm=llm, 
    verbose=True,
    allow_parallel_tool_calls=False
)

# Initialize the structured planner agent with the worker
structured_planner = StructuredPlannerAgent(
    worker, 
    tools=[query_engine_tool], 
    verbose=True
)

# Test with a complex question
response = structured_planner.chat("What are the names of the files contianing information about Juan Maldacena according to the metadata?")
display_citation(response.response)

## Document Retrieval Tool

Now, let's give the agent a document retrieval tool, once it found the file location, it now has the option to add complete file pages to its context, in order to learn more about the data source.

In [67]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader

# Define the directory containing documents
source_dir = Path("assets", "extraction", "markdown")

# Load markdown documents
documents = SimpleDirectoryReader(source_dir, required_exts=[".md"], recursive=True).load_data()
print(f"Loaded {len(documents)} documents")


Loaded 16 documents


In [85]:
from pathlib import Path
from typing import Any
from llama_index.core.tools import BaseTool, ToolMetadata

class ListDocumentsTool(BaseTool):
    def __init__(self, documents):
        """
        """
        self.documents = documents
        super().__init__()

    @property
    def metadata(self) -> ToolMetadata:
        return ToolMetadata(
            name="ListDocuments",
            description="Lists all document titles derived from the parent folder of each file's path or source."
        )

    def run(self, **kwargs) -> str:
        results = []
        for doc in self.documents:
            # Try to get the file path from metadata; fallback to 'source'
            file_path = doc.metadata.get("file_path") or doc.metadata.get("source")
            if file_path:
                # Extract the title from the parent folder's stem.
                title = Path(file_path).parent.stem
            else:
                title = "Unknown Document"
            results.append(title)
        return "\n".join(results)

    def __call__(self, input: Any) -> str:
        # This tool ignores the input and simply lists document titles.
        return self.run()

In [81]:
list_tool = ListDocumentsTool(documents)
print(list_tool.run())

de-alpha-beta-gamma-paper
markdown
de-bell-labs
de-edward-witten
de-feynman
de-juan-maldacena
eng-ads-cft-correspondence
eng-calabi-yao-manifolds
eng-feynman-the-development-of-the-qed-space-time-view
eng-looking-for-a-bulk-point
eng-m-theory
eng-manifestly-supersymmetric-m-theory
eng-quantum-electro-dynamics
eng-replica-wormholes-and-the-entropy-of-hawking-radiation
eng-strings-on-pp-waves-and-massives-two-dimensional-field-theories
fr-losalamos


In [ ]:
from llama_index.core.tools import FunctionTool, ToolOutput


class RetrieveDocumentTool(BaseTool):
    def __init__(self, documents):
        """
        """
        self.documents = documents
        super().__init__()

    @property
    def metadata(self) -> ToolMetadata:
        return ToolMetadata(
            name="RetrieveDocument",
            description="Retrieves the full text of a document given its title derived from its file path or source."
        )

    def run(self, title: str, **kwargs) -> str:
        for doc in self.documents:
            file_path = doc.metadata.get("file_path") or doc.metadata.get("source")
            if file_path:
                doc_title = Path(file_path).parent.stem
                if doc_title == title:
                    return doc.text  # or doc.content, depending on your document structure
        return f"Document with title '{title}' not found."

    def __call__(self, input: Any) -> str:
        # Expect input to be a dict with a "title" key, or directly a title string.
        if isinstance(input, dict):
            title = input.get("title")
        else:
            title = input
        return self.run(title=title)

In [83]:
retrieve_tool = RetrieveDocumentTool(documents)
title_to_retrieve = "de-juan-maldacena"
display_citation(retrieve_tool.run(title=title_to_retrieve))

> ![](_page_0_Picture_2.jpeg)
> 
> # Juan Maldacena
> 
> Juan Martín Maldacena (born 10 September 1968) is an Argentine theoretical physicist and the Carl P. Feinberg Professor in the School of Natural Sciences at the Institute for Advanced Study, Princeton. 3 He has made significant contributions to the foundations of string theory and quantum gravity. His most famous discovery is the AdS/CFT correspondence, a realization of the holographic principle in string theory.
> 
> # Biography
> 
> ![](_page_0_Picture_6.jpeg)
> 
> Instituto Balseiro at which Maldacena obtained his Physics licenciatura
> 
> Maldacena obtained his licenciatura (a six-year degree) in 1991 at the Instituto Balseiro, Bariloche, Argentina, under the supervision of Gerardo Aldazábal. He then obtained his Ph.D. in physics at Princeton University after completing a doctoral dissertation titled "Black holes in string theory" under the supervision of Curtis Callan in 1996, and went on to a post-doctoral position at Rutgers University. In 1997, he joined Harvard University as associate professor, being quickly promoted to Professor of Physics in 1999. Since 2001 he has been a professor at the Institute for Advanced Study in Princeton, New Jersey and in 2016 became the first Carl P. Feinberg Professor of Theoretical Physics in the institute's School of Natural Sciences.
> 
> #### Juan Maldacena
> 
> ![](_page_0_Picture_10.jpeg)
> 
> | Born              | 10 September 1968                                                         |  |  |
> |-------------------|---------------------------------------------------------------------------|--|--|
> |                   | Buenos Aires, Argentina                                                   |  |  |
> | Nationality       | Argentine, American, Italian121                                           |  |  |
> | Education         | Balseiro Institute (BS)                                                   |  |  |
> |                   | Princeton University (PhD)                                                |  |  |
> | Known for         | AdS/CFT correspondence<br>ER = EPR<br>ABJM superconformal field<br>theory |  |  |
> | Awards            | Alfred P. Sloan Foundation<br>Fellowship                                  |  |  |
> |                   | Sackler Prize                                                             |  |  |
> |                   | MacArthur Fellowship                                                      |  |  |
> |                   | Xanthopoulos                                                              |  |  |
> |                   | Pius XI Medal                                                             |  |  |
> |                   | Dirac Medal                                                               |  |  |
> |                   | Pomeranchuk Prize                                                         |  |  |
> |                   | Breakthrough Prize in                                                     |  |  |
> |                   | Fundamental Physics                                                       |  |  |
> |                   | Lorentz Medal                                                             |  |  |
> |                   | Albert Einstein Medal                                                     |  |  |
> |                   | St. Albert Award                                                          |  |  |
> |                   | Galileo Galilei Medal                                                     |  |  |
> | Scientific career |                                                                           |  |  |
> | Fields            | Theoretical physics                                                       |  |  |
> 
> | Maldacena is a member of the Society of Catholic | Institutions | Institute for Advanced Study    |
> |--------------------------------------------------|--------------|---------------------------------|
> | Scientists. 41                                   | Thesis       | Black Holes in String Theory 11 |
> |                                                  |              | (1996)                          |
> |                                                  | Doctoral     | Curtis Callan                   |
> |                                                  | advisor      |                                 |
> | Contributions to physics                         |              |                                 |
> 
> Maldacena has made numerous discoveries in theoretical physics. Leonard Susskind called him "perhaps the greatest physicist of his generation ... certainly the greatest theoretical physicist of his generation".[5] His most famous discovery is the most reliable realization of the holographic principle - namely the AdS/CFT correspondence, a conjecture about the equivalence of string theory on Anti-de Sitter (AdS) space, and a conformal field theory defined on the boundary of the AdS space.[0] According to the conjecture, certain theories of quantum gravity are equivalent to other quantum mechanical theories (with no gravitational force) in one fewer spacetime dimensions.
> 
> In subsequent works, Maldacena elucidated several aspects of the AdS/CFT correspondence, describing how certain physical observables defined in one theory can be described in the equivalent theory. Shortly after his original work on the AdS/CFT correspondence, Maldacena showed how Wilson lines can be computed in a corresponding string theory by considering the area swept by an evolving fundamental string.[2] Wilson lines are non-local physical observables defined in gauge theory. In 2001, Maldacena proposed that an eternal black hole, an object defined in a gravitational theory, is equivalent to a certain entangled state involving two copies of the corresponding quantum mechanical theory.[ Ordinary black holes emit Hawking radiation and eventually evaporate. An eternal black hole is a type of black hole that survives forever because it eventually re-absorbs the radiation it emits.
> 
> In 2013, Maldacena co-authored an analysis of the 2012 black hole firewall paradox with Leonard Susskind, arguing that the paradox can be resolved if entangled particles are connected by minor wormholes."[9][10][11]
> 
> ### Publications
> 
> 3/5/25, 10:40 AM
> 
> ■ Berenstein, David; Maldacena, Juan; Nastase, Horatiu (2002). "Strings in flat space and pp waves from N = 4 Super Yang Mills". AIP Conference Proceedings. 646. Waterloo, Ontario (Canada): 3-14. arXiv:hep-th/0202021 (https://arxiv.org/abs/hep-th/0202021). Bibcode:2002AIPC..646....3B (https://ui.adsabs.harvard.edu/abs/2002AIPC..646....3B). doi:10.1063/1.1524550 (https://doi.org/10.1063%2F1.1524550).
> 
> # Awards
> 
> Maldacena has received these awards:
> 
> - Alfred P. Sloan Foundation Fellowship, 1998
> - Packard Fellowship in Science and Engineering, 1998
> - MacArthur Fellowship, 1999[12]
> - UNESCO Husein Prize for Young Scientists, 1999
> - · Sackler Prize in Physics, 2000
> - Xanthopoulos International Award for Research in Gravitational Physics, 20011131
> - Pius XI Medal, 2002
> - = Edward A. Bouchet Award 141 of the American Physical Society, 2004
> - = Member of the American Academy of Arts and Sciences, elected 2007.155
> - Dannie Heineman Prize, 2007[16]
> - Dirac Medal of the ICTP, 2008
> - Pomeranchuk Prize, 2012
> - Breakthrough Prize in Fundamental Physics, 2012 [17]
> - = Member of the National Academy of Sciences, elected 2013[18]
> - Diamond Konex Award as the most important scientist in the last decade in Argentina, 2013
> - Lorentz Medal, 2018[19]
> - Albert Einstein Medal, 2018[20]
> - St. Albert Award, 2018[21]
> - = Galileo Galilei Medal, 2019
> - = Les Houches School of Physics prize 2020[22]
> 
> ### References
> 
> - 1. Maldacena, Juan M. (June 1996), "Black Holes in String Theory (PhD Thesis)". arXiv:hepth/9607235 (https://arxiv.org/abs/hep-th/9607235). Princeton University
> - 2. "Curriculum Vitae: Juan Maldacena" (https://www.ias.edu/sites/default/files/sns/files/Maldacena CV2020.pdf) (PDF). 2020.
> - 3. "Juan Maldacena" (https://www.ias.edu/scholars/maldacena). Institute for Advanced Study. 9 December 2019. Retrieved 13 May 2020.
> - 4. "Juan Martín Maldacena" (https://catholicscientists.org/st-albert-award/juan-martin-maldacen al). The Society of Catholic Scientists. Retrieved 3 November 2024.
> - 5. Susskind, Leonard (30 April 2014). Black Holes, the Conservation of Information, and the Holographic Principle (https://www.cornell.edu/video/leonard-susskind-2-black-holes-conservati on-of-information-holographic-principle) (Speech). Messenger Lectures on "The Birth of the Universe and the Origin of Laws of Physics". Cornell University. 68 minutes in.
> - 6. Juan Martin Maldacena (1998). "The Large N Limit of Superconformal Field Theories and Supergravity". Adv. Theor. Math. Phys. 2 (2): 231–252. arXiv:hep-th/9711200 (https://arxiv.org/ abs/hep-th/9711200). Bibcode:1998AdTMP...2..231M (https://ui.adsabs.harvard.edu/abs/1998A dTMP...2..231M), doi:10.4310/atmp.1998.v2.n2.a1 (https://doi.org/10.4310%2Fatmp.1998.v2.n 2.a1).
> - 7. Maldacena, Juan M. (1 June 1998). "Wilson loops in field theories". Physical Review Letters: 80 (22): 4859-4862. arXiv:hep-th/9803002 (https://arxiv.org/abs/hep-th/9803002), Bibcode:1998PhRvL..80.4859M (https://ui.adsabs.harvard.edu/abs/1998PhRvL..80.4859M). doi:10.1103/PhysRevLett.80.4859 (https://doi.org/10.1103%2FPhysRevLett.80.4859). ISSN 0031-9007 (https://search.worldcat.org/issn/0031-9007). S2CID 14386667 (https://api.se manticscholar.org/CorpusID:14386667).
> - 8. Maldacena, Juan M. (12 April 2003). "Eternal Black Holes in AdS". Journal of High Energy Physics. 2003 (4): 021. arXiv:hep-th/0106112 (https://arxiv.org/abs/hep-th/0106112). doi:10.1088/1126-6708/2003/04/021 (https://doi.org/10.1088%2F1126-6708%2F2003%2F04% 2F021). ISSN 1029-8479 (https://search.worldcat.org/issn/1029-8479). S2CID 7767700 (http s://api.semanticscholar.org/CorpusID:7767700).
> - 9. Overbye, Dennis (12 August 2013). "A Black Hole Mystery Wrapped in a Firewall Paradox" (htt ps://www.nytimes.com/2013/08/13/science/space/a-black-hole-mystery-wrapped-in-a-firewall-p aradox.html). New York Times. Retrieved 29 October 2013.
> - 10. "The Firewall Paradox" (https://www.nytimes.com/interactive/2013/08/13/science/0813-sci-blac khole.html). New York Times. 12 August 2013. Retrieved 29 October 2013.
> - 11. Cool horizons for entangled black holes
> - 12. "Meet the 1999 MacArthur Fellows" (http://www.macfound.org/fellows/class/july-1999/). Retrieved 25 November 2013.
> - 13. "Basilis Xanthopoulos International Award" (https://www.physics.uoc.gr/Xanthopoulos/). www.physics.uoc.gr
> - 14. "2004 Edward A. Bouchet Award Recipient: Juan M. Maldacena" (http://www.aps.org/program s/honors/prizes/prizerecipient.cfm?first\_nm=Juan&last\_nm=Maldacena&year=2004). Retrieved 24 August 2011.
> - 15. "Juan Martin Maldacena" (https://www.amacad.org/person/juan-martin-maldacena). American Academy of Arts & Sciences. Retrieved 13 May 2020.
> - 16. "2007 Dannie Heineman Prize for Mathematical Physics Recipient: Juan Maldacena, Institute for Advance Study" (https://www.aps.org/programs/honors/prizes/prizerecipient.cfm?last\_nm= Maldacena&first\_nm=Juan&year=2007). American Physical Society. Retrieved 3 August 2018.
> - 17. New annual US\$3 million Fundamental Physics Prize recognizes transformative advances in the field (https://breakthroughprize.org/News/15) Archived (https://web.archive.org/web/201208 03211628/https://breakthroughprize.org/Prize/1) 3 August 2012 at the Wayback Machine, FPP, accessed 1 August 2012
> - 18. "Juan Maldacena" (http://www.nasonline.org/members/20022805.html). www.nasonline.org. Retrieved 13 May 2020.
> - 19. "LORENTZ MEDAL AWARDED TO PHYSICIST JUAN MARTÍN MALDACENA" (https://web.ar chive.org/web/20210720163807/https://www.knaw.nl/en/news/lorentz-medal-awarded-tophysicist-juan-martin-maldacena). Royal Netherlands Academy of Arts and Sciences. 27 March 2018. Archived from the original (https://www.knaw.nl/en/news/lorentz-medal-awarded-tophysicist-juan-martin-maldacena) on 20 July 2021. Retrieved 3 August 2018.
> - 20. "Juan Maldacena Receives 2018 Einstein Medal from the Albert Einstein Society IAS News | Institute for Advanced Study" (https://www.ias.edu/news/juan-maldacena-receives-2018-einstei n-medal-albert-einstein-society). 3 August 2018.
> - 21. "Juan Martin Maldacena" (https://web.archive.org/web/20210118091002/https://www.catholicsc ientists.org/about/St.-Albert-Award/Maldacena-bio). Archived from the original (https://www.cat holicscientists.org/about/St.-Albert-Award/Maldacena-bio) on 18 January 2021. Retrieved 6 October 2018.
> - 22. Peyla, Philippe. "2020 Prize: Juan Martin Maldacena" (https://www.houches-school-physics.co m/prize/2020-prize/2020-prize-juan-martin-maldacena-764797.kjsp?RH=1606314527588). Ecole des Houches, Retrieved 20 December 2023,
> 
> ### External links
> 
> - Maldacena's web page at the Institute (http://www.sns.ias.edu/~malda)
> - Maldacena Theme tree (http://xstructure.inr.ac.ru/x-bin/search e3.py?au=Maldacena J&start= 0&max results=100)
> - Interview of Juan Maldacena by David Zierler on January 15, 2021, Niels Bohr Library & Archives, American Institute of Physics, College Park, MD USA (https://www.aip.org/history-pro grams/niels-bohr-library/oral-histories/47184)
> 
> Retrieved from "https://en.wikipedia.org/w/index.php?title=Juan\_Maldacena&oldid=1267298756"

## Integrating the Tools into Agent

We can now introduce the additional tools into the agent:

In [84]:
import nest_asyncio
from llama_index.core.agent import StructuredPlannerAgent, FunctionCallingAgentWorker

# Ensure async loops work in notebooks
nest_asyncio.apply()

# The worker takes the list of tools (your custom ones) and the LLM.
worker = FunctionCallingAgentWorker.from_tools(
    [list_tool, retrieve_tool],
    llm=llm,
    verbose=True,
    allow_parallel_tool_calls=False
)

# Using a structured planner agent that leverages our worker and tools.
agent = StructuredPlannerAgent(
    worker,
    tools=[list_tool, retrieve_tool],
    verbose=True
)

# Example: Ask the agent to list documents, then retrieve one.
query = (
    "First, list all document titles available. "
    "Then, retrieve the document with title 'example_title'."
)
response = agent.chat(query)
print("Agent response:\n", response)


=== Initial plan ===
List Document Titles:
Use the ListDocuments tool to list all document titles available. -> A list of all document titles.
deps: []


Retrieve Specific Document:
Use the RetrieveDocument tool to retrieve the document with the title 'example_title'. -> The full text of the document with the title 'example_title'.
deps: ['List Document Titles']


> Running step b4a50ce1-bd43-43a5-908a-6748dc861db9. Step input: Use the ListDocuments tool to list all document titles available.
Added user message to memory: Use the ListDocuments tool to list all document titles available.
=== Calling Function ===
Calling function: ListDocuments with args: {"input": "List all document titles"}
=== Function Output ===


AttributeError: 'str' object has no attribute 'content'